# Nemotron ColEmbed V2 — batch page indexing (Phase 2)

**Purpose**: embed every page of the 7-file corpus (172p) and the 60 benchmark queries **inside Colab**,
save fp16 arrays, download one `embeddings.zip`. No tunnel, no per-page HTTP round trip.
Local side: `scripts/13_load_nemotron_npz.py` upserts the zip into Qdrant (`pharma_vision`, gRPC).

Replaces `02_nemotron_tunnel.ipynb` for indexing (tunnel stays only as a fallback for ad-hoc query embedding).

## How to run
1. Runtime → Change runtime type → **T4 GPU** (free tier is enough; ~15 min for 172p).
2. Left sidebar 🔑 → add `HF_TOKEN` secret.
3. Run cells top to bottom. Cell 3 opens an upload dialog: select the **7 corpus PDFs**
   (`Q1.pdf Q2.pdf Q3.pdf 20F_extract.pdf Q1_deck.pdf Q2_deck.pdf Q3_deck.pdf`) **and** `eval/questions.jsonl`.
4. Last cell downloads `embeddings.zip` (~2.4 GB fp16; Colab download may take a few minutes).
   If the browser download stalls, run the optional Drive cell instead and fetch from Drive.

## Output layout inside embeddings.zip
```
manifest.json                      # {model_id, render_scale, pages:[{source,page,n_patches}], queries:[...]}
pages/<source>/<page>.npy          # fp16 [N_patches, 3072], page is 1-based, source is the PDF filename
queries/<id>_ko.npy, <id>_en.npy   # fp16 [N_tokens, 3072]
```
Re-running is resume-safe: existing `.npy` files are skipped.


## 1. Install + HF auth

In [ ]:
!pip install -q transformers accelerate einops sentencepiece pypdfium2 Pillow

import os
from huggingface_hub import login
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])


## 2. Load Nemotron ColEmbed V2 (bf16 + float-only cast, same as notebook 02)

In [ ]:
import torch
from transformers import AutoModel

MODEL_ID = 'nvidia/llama-nemotron-colembed-vl-3b-v2'   # T4-friendly (ViDoRe V3 rank 6)
RENDER_SCALE = 1.5      # ~150 DPI; the tunnel path used 0.85 to survive Cloudflare limits — not needed here
PAGE_BATCH = 2          # images per forward_images call on T4 (16 GB); raise to 4 on L4/A100

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'Select a GPU runtime first'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'{torch.cuda.get_device_name(0)}  VRAM {vram_gb:.1f} GB')

model = AutoModel.from_pretrained(
    MODEL_ID, device_map=device, trust_remote_code=True, torch_dtype=torch.bfloat16,
).eval()
FLOAT_DTYPES = (torch.float32, torch.float16, torch.bfloat16, torch.float64)
for p in model.parameters():
    if p.dtype in FLOAT_DTYPES:
        p.data = p.data.to(torch.bfloat16)
for b in model.buffers():
    if b.dtype in FLOAT_DTYPES:
        b.data = b.data.to(torch.bfloat16)
print(f'Loaded {MODEL_ID}: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params')


def as_list(out):
    """forward_images / forward_queries -> list of per-item tensors."""
    if isinstance(out, list):
        return out
    if hasattr(out, 'dim') and out.dim() == 3:
        return [out[i] for i in range(out.shape[0])]
    return [out]


## 3. Upload corpus PDFs + questions.jsonl (multi-select in the dialog)

In [ ]:
from pathlib import Path
from google.colab import files

IN = Path('/content/input'); IN.mkdir(exist_ok=True)
uploaded = files.upload()
for name, data in uploaded.items():
    (IN / name).write_bytes(data)

CORPUS = ['Q1.pdf', 'Q2.pdf', 'Q3.pdf', '20F_extract.pdf', 'Q1_deck.pdf', 'Q2_deck.pdf', 'Q3_deck.pdf']
missing = [n for n in CORPUS + ['questions.jsonl'] if not (IN / n).exists()]
assert not missing, f'missing uploads: {missing}'
print('input ok:', sorted(p.name for p in IN.iterdir()))


## 4. Embed every page → `pages/<source>/<page>.npy` (fp16). Resume-safe.

In [ ]:
import json, time
import numpy as np
import pypdfium2 as pdfium

OUT = Path('/content/embeddings'); (OUT / 'pages').mkdir(parents=True, exist_ok=True); (OUT / 'queries').mkdir(exist_ok=True)
manifest = {'model_id': MODEL_ID, 'render_scale': RENDER_SCALE, 'dim': 3072, 'pages': [], 'queries': []}


def render_pages(pdf_path):
    pdf = pdfium.PdfDocument(str(pdf_path))
    try:
        for i in range(len(pdf)):
            yield i + 1, pdf[i].render(scale=RENDER_SCALE).to_pil().convert('RGB')
    finally:
        pdf.close()


t0 = time.time(); done = 0
for source in CORPUS:
    out_dir = OUT / 'pages' / source; out_dir.mkdir(parents=True, exist_ok=True)
    batch = []
    def flush():
        global done
        if not batch:
            return
        imgs = [im for _, im in batch]
        with torch.no_grad():
            embs = as_list(model.forward_images(imgs, batch_size=len(imgs)))
        for (page, _), emb in zip(batch, embs):
            arr = emb.detach().to(torch.float16).cpu().numpy()
            np.save(out_dir / f'{page}.npy', arr)
            manifest['pages'].append({'source': source, 'page': page, 'n_patches': int(arr.shape[0])})
            done += 1
        batch.clear()
    for page, img in render_pages(IN / source):
        f = out_dir / f'{page}.npy'
        if f.exists():
            arr = np.load(f, mmap_mode='r')
            manifest['pages'].append({'source': source, 'page': page, 'n_patches': int(arr.shape[0])}); done += 1
            continue
        batch.append((page, img))
        if len(batch) >= PAGE_BATCH:
            flush()
    flush()
    print(f'{source}: done  (total {done} pages, {time.time()-t0:.0f}s)')

n_patch = [p['n_patches'] for p in manifest['pages']]
print(f'pages {len(n_patch)}  patches/page min {min(n_patch)} mean {sum(n_patch)/len(n_patch):.0f} max {max(n_patch)}')


## 5. Embed the 60 benchmark queries (30 × ko/en) → `queries/<id>_<lang>.npy`

In [ ]:
qs = [json.loads(l) for l in (IN / 'questions.jsonl').read_text(encoding='utf-8').splitlines() if l.strip()]
items = [(f"{q['id']}_ko", q['q_ko']) for q in qs] + [(f"{q['id']}_en", q['q_en']) for q in qs]
texts = [t for _, t in items]
with torch.no_grad():
    embs = as_list(model.forward_queries(texts, batch_size=8))
for (name, text), emb in zip(items, embs):
    arr = emb.detach().to(torch.float16).cpu().numpy()
    np.save(OUT / 'queries' / f'{name}.npy', arr)
    manifest['queries'].append({'id': name, 'n_tokens': int(arr.shape[0]), 'text': text})
print(f'{len(items)} queries embedded')


## 6. Zip + download

In [ ]:
import shutil
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=1), encoding='utf-8')
zip_path = shutil.make_archive('/content/embeddings', 'zip', OUT)
print(zip_path, f'{Path(zip_path).stat().st_size/1e9:.2f} GB')
files.download(zip_path)


### (optional) copy to Google Drive instead of browser download

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# shutil.copy('/content/embeddings.zip', '/content/drive/MyDrive/embeddings.zip')
